# Week 5: Model a magnetic dipole from surface data

**Practical exercise, 25–35 minutes.** Use projection, superposition, and residuals to estimate a dipole and one additional field component. No spherical-harmonic software or theory is required; optional investigations can take longer.

## Data and physical setting

The CSV file contains 2,048 model-generated values of Earth's outward radial magnetic field. They come from IGRF-14, a reference model based on observations, using degrees 1–13 for epoch 2025.0. The values are calculated assuming a spherical Earth with radius $a=6371.2\,\mathrm{km}$; they are not direct measurements.

The columns are geographic colatitude `theta_geo_deg`, east longitude `longitude_east_deg`, geocentric latitude `latitude_geocentric_deg`, outward radial field `Br_uT` in microtesla, and solid-angle weight `weight_sr`. Colatitude is measured from geographic north; latitude is $90^\circ-\theta_g$ and is not geodetic latitude. The grid uses 32 Gauss–Legendre colatitudes and 64 evenly spaced longitudes. The weights represent each sample's share of the sphere and sum to $4\pi$ steradians. Use them for surface sums: do not multiply by $\sin\theta_g$ again or take a plain average.

The accompanying metadata gives the epoch, reference radius, coefficient provenance, and unit vector along the reference dipole moment. Its direction is supplied for the main exercise, so estimate the strength. Geographic $x$ points to latitude $0^\circ$, longitude $0^\circ$; $y$ points to latitude $0^\circ$, longitude $90^\circ\mathrm{E}$ ; and $z$ points to geographic north. The data and metadata files must remain beside this notebook. No answer coefficients are included.

In [ ]:
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt

folder = Path('.')  # run with this notebook folder as the working directory
samples = np.genfromtxt(folder / 'igrf14_surface_2025.csv', delimiter=',', names=True)
metadata = json.loads((folder / 'metadata.json').read_text())
theta = np.deg2rad(samples['theta_geo_deg'])
phi = np.deg2rad(samples['longitude_east_deg'])
br = samples['Br_uT']
w = samples['weight_sr']
mhat = np.array(metadata['dipole_axis_unit_vector_xyz'])
print('Number of values:', br.size)
print('Sum of weights:', w.sum(), 'Expected:', 4*np.pi)


In [ ]:
def map_values(values, title, limit=None):
    if limit is None:
        limit = np.max(np.abs(values))
    fig, ax = plt.subplots(figsize=(10, 3.5))
    points = ax.scatter(samples['longitude_east_deg'], samples['latitude_geocentric_deg'],
                        c=values, cmap='RdBu_r', vmin=-limit, vmax=limit, s=13)
    ax.set(xlabel='Longitude east (degrees)', ylabel='Geocentric latitude (degrees)', title=title)
    fig.colorbar(points, ax=ax, label='Radial field (µT), outward positive')
    plt.show()

map_values(br, 'IGRF-14 reference, 2025.0')


## 1. Build the dipole pattern

Convert the geographic angles to radians and construct the outward unit vector

$$
\hat r=(\sin\theta_g\cos\varphi,\ \sin\theta_g\sin\varphi,\ \cos\theta_g).
$$

Use the supplied unit vector of the dipole moment, $\hat m$, to calculate the dipole pattern

$$
f_d=\hat m\cdot\hat r=\cos\theta_m.
$$

Explain why geographic colatitude $\theta_g$ and angle from the dipole axis $\theta_m$ are different. State the sign of $f_d$ at each end of the magnetic axis. For a centered dipole, the radial field at the sphere is

$$
B_{r,d}=\alpha f_d,\qquad \alpha=\frac{2C}{a^3},\qquad C=\frac{\mu_0|\vec m|}{4\pi}.
$$

The pattern is a cosine: it equals $+1$ along $\hat m$, $-1$ in the opposite direction, and zero perpendicular to the axis.

In [ ]:
# Build one outward unit vector per sample, with shape (number_of_samples, 3).
rhat = None

# Obtain one value per surface point (number_of_samples,)
fd = None

## 2. Project the surface field onto the dipole pattern

The surface inner product is $\langle u,v\rangle=\int uv\,d\Omega$, where $d\Omega=\sin\theta_g\,d\theta_g\,d\varphi$. Approximate it with the supplied weights:

$$
\alpha=\frac{\langle B_r,f_d\rangle}{\langle f_d,f_d\rangle}
\simeq\frac{\sum_i w_i B_{r,i}f_{d,i}}{\sum_i w_i f_{d,i}^2}.
$$

Calculate $\alpha$ in $\mu\mathrm{T}$. Explain why the formula divides by the squared size of the pattern, and compare this with finding a Fourier coefficient. If you reverse the chosen axis, how do $f_d$ and $\alpha$ change? Does the fitted field change?

Use $\alpha$ to find $C$ in T $\mathrm{m}^3$, and optionally the moment magnitude, $A$, in $\mathrm{m}^2$. Convert $\alpha$ from $\mu\mathrm{T}$ to T and $a=6371.2\,\mathrm{km}$ to metres first. Since $C$ is a magnitude, use $C=|\alpha|a^3/2$ if $\alpha$ is negative. Is $\alpha$ the radial field strength at the magnetic pole or equator? For the moment magnitude, use $|\vec m|=4\pi C/\mu_0$.

In [ ]:
# TODO: calculate the projection using the supplied weights.
alpha = None  # µT
# TODO: recover C in T m^3, and optionally |m| in A m^2.


## 3. What does the dipole model leave out?

Calculate the fitted radial field $B_{r,d}=\alpha f_d$ and residual $B_r-B_{r,d}$. Plot the observed field and fitted field with the same colour limits, then plot the residual with its own scale so its structure is visible. Calculate the area-weighted RMS residual,

$$
\mathrm{RMS}(q)=\sqrt{\frac{\sum_i w_i q_i^2}{\sum_i w_i}}.
$$

Verify that the residual inner product with $f_d$ is approximately zero. Does that imply that the residual vanishes everywhere? Compare fitting a coefficient to checking the model at one location: projection fits the full surface in a weighted least-squares sense, but does not require the dipole to meet the data at every point.

In [ ]:
# TODO: calculate fit, residual and weighted RMS.
# Use map_values(...) for plots.


## 4. Add an aligned axial quadrupole

Use the same magnetic axis to define a second pattern, the so-called "aligned axial quadrupole":

$$
f_q=\frac{3f_d^2-1}{2},\qquad B_r\approx\alpha f_d+\beta f_q.
$$

Check whether $\sum_i w_i f_{d,i}f_{q,i}$ is close to zero. Explain the symmetry: at opposite points, the dipole pattern changes sign, but the quadrupole pattern keeps the same value. Project both the data and the dipole residual onto $f_q$ to find $\beta$. Why should these two estimates agree? Plot the residual after adding the quadrupole and compare its weighted RMS with the dipole-only value.

For the potential used in class, $\Psi_q=D f_q/r^3$, calculate $D=\beta a^4/3$ in T m$^4$. Convert $\beta$ and $a$ to SI units first. Use the sign of $\beta$ to explain how the quadrupole changes the field at the two ends of the magnetic axis.

This fits one axisymmetric quadrupole pattern about the supplied magnetic axis. It is not the full degree-two field, which has five independent patterns, and it is not generally the geographic-axis IGRF coefficient $g_2^0$. Other parts of the field are therefore expected to remain in the residual.

In [ ]:
fq = None     # TODO
beta = None   # TODO, µT
# TODO: recover D in T m^4 and calculate the new residual.


## Optional investigations

### Estimate the dipole direction too

Instead of using the supplied axis, use the three geographic patterns

$$
f_x=\sin\theta_g\cos\varphi,\qquad f_y=\sin\theta_g\sin\varphi,\qquad f_z=\cos\theta_g.
$$

Project $B_r$ onto each pattern using the full-sphere weights, with the appropriate pattern norm in each denominator. Assemble the three coefficients into a vector. Its length is the dipole polar amplitude; its unit direction estimates the moment direction. Compare that direction with the metadata. This recovers the full centred tilted dipole from radial field alone.

### Use data from only part of Earth

Keep only samples in the geographic northern hemisphere. Recheck pattern orthogonality and compare independent projections with a simultaneous weighted least-squares fit to your chosen patterns. Explain why full-sphere orthogonality does not guarantee independent coefficient recovery from incomplete or unevenly weighted observations. Omitted spatial patterns can bias even the simultaneous limited-area fit.

## Sources and reproducibility

- IAGA/NOAA [IGRF-14 coefficients](https://www.ngdc.noaa.gov/IAGA/vmod/coeffs/igrf14coeffs.txt).
- [IGRF-14 model description](https://doi.org/10.1186/s40623-025-02360-0).

The course generator uses the same Schmidt semi-normalized synthesis as the TeachBook. This dataset is at a fixed reference epoch and sphere; it is not a claim about current measurements or ellipsoidal ground level. Full provenance and the coefficient-file checksum are recorded in `metadata.json`.